# MiniMax-M2 Tool Calling (Araç Kullanımı)

Bu notebook, MiniMax-M2'nin araç/fonksiyon çağırma özelliklerini gösterir.

MiniMax-M2, karmaşık araç kullanımı ve hata kurtarma konusunda mükemmeldir.

## İçindekiler
1. Araç Tanımlama
2. Araç Çağrısı
3. Çoklu Araç Kullanımı
4. Araç Sonuçlarını İşleme

In [ ]:
import json
from openai import OpenAI

# Yapılandırma
client = OpenAI(
    api_key="your-api-key",
    base_url="http://localhost:8000/v1",
)
MODEL_NAME = "MiniMax-M2"

## 1. Araç Tanımlama

Araçlar JSON Schema formatında tanımlanır.

In [ ]:
# Araç tanımları
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Belirtilen şehrin hava durumunu getirir",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Şehir adı, örn: Istanbul"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Sıcaklık birimi"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Matematiksel hesaplama yapar",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Hesaplanacak ifade, örn: 2 + 2"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

print("Tanımlanan araçlar:")
for tool in tools:
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")

## 2. Araç Çağrısı

In [ ]:
# Model ile araç çağrısı
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "İstanbul'da hava nasıl?"}
    ],
    tools=tools,
    tool_choice="auto",  # Otomatik araç seçimi
)

message = response.choices[0].message

# Araç çağrısı var mı kontrol et
if message.tool_calls:
    for tool_call in message.tool_calls:
        print(f"Araç: {tool_call.function.name}")
        print(f"Argümanlar: {tool_call.function.arguments}")
else:
    print("Araç çağrısı yok")
    print(f"Yanıt: {message.content}")

## 3. Araç Fonksiyonlarını Uygulama

In [ ]:
# Gerçek araç fonksiyonları
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Simüle edilmiş hava durumu API'si"""
    # Gerçek uygulamada bir API çağrısı yapılır
    weather_data = {
        "Istanbul": {"temp": 22, "condition": "Güneşli"},
        "Ankara": {"temp": 18, "condition": "Bulutlu"},
        "Izmir": {"temp": 28, "condition": "Açık"},
    }
    
    data = weather_data.get(city, {"temp": 20, "condition": "Bilinmiyor"})
    
    if unit == "fahrenheit":
        data["temp"] = data["temp"] * 9/5 + 32
    
    return {
        "city": city,
        "temperature": data["temp"],
        "unit": unit,
        "condition": data["condition"]
    }

def calculate(expression: str) -> dict:
    """Güvenli hesaplama"""
    try:
        # Sadece güvenli karakterlere izin ver
        allowed = set("0123456789+-*/.() ")
        if all(c in allowed for c in expression):
            result = eval(expression)
            return {"expression": expression, "result": result}
        return {"error": "Geçersiz ifade"}
    except Exception as e:
        return {"error": str(e)}

# Araç registry
TOOLS = {
    "get_weather": get_weather,
    "calculate": calculate,
}

## 4. Tam Döngü: İstek → Araç → Yanıt

In [ ]:
def process_with_tools(user_message: str) -> str:
    """Araç kullanarak kullanıcı mesajını işle"""
    
    messages = [{"role": "user", "content": user_message}]
    
    # İlk istek
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    
    message = response.choices[0].message
    
    # Araç çağrısı yoksa direkt yanıt döndür
    if not message.tool_calls:
        return message.content
    
    # Asistan mesajını geçmişe ekle
    messages.append(message)
    
    # Her araç çağrısını işle
    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        print(f"🔧 Araç çağrılıyor: {func_name}({func_args})")
        
        # Fonksiyonu çalıştır
        if func_name in TOOLS:
            result = TOOLS[func_name](**func_args)
        else:
            result = {"error": f"Bilinmeyen araç: {func_name}"}
        
        print(f"📋 Sonuç: {result}")
        
        # Sonucu mesajlara ekle
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })
    
    # Final yanıt
    final_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
    )
    
    return final_response.choices[0].message.content

# Test
result = process_with_tools("İstanbul'da hava nasıl ve 25 + 17 kaç eder?")
print(f"\n🤖 Final Yanıt:\n{result}")

## Sonuç

Bu notebook'ta öğrendikleriniz:

- ✅ JSON Schema ile araç tanımlama
- ✅ Araç çağrılarını algılama
- ✅ Araç fonksiyonlarını uygulama
- ✅ Sonuçları modele geri gönderme

MiniMax-M2, karmaşık araç zincirleri ve hata kurtarma konusunda mükemmeldir!